# Objetivo
Testar biblioteca para cálculo de ofensores de chamadas abusivas. 

Neste notebook serão testadas, sobre uma amostra dos dados de CDRs, funções para:

- extrair CDRs processados do formato texto para parquet;
- transformar CDRs para formato normalizado (tabela única com campos uniformes);
- carregar e calcular chamadas curtas;

> Atenção: este notebook deve ser executado no kernel com Python 3.9

# Configuração do ambiente

## Bibliotecas

In [ ]:
import logging

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from teleutils import robocalls

## Logging

In [ ]:
# Configuração mínima para exibir logs no output da célula
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
)

# Opcional: reduzir ruído de bibliotecas externas (pyspark, py4j, etc.)
logging.getLogger("py4j").setLevel(logging.WARNING)
logging.getLogger("pyspark").setLevel(logging.WARNING)

## Spark Session

In [ ]:
spark = SparkSession.builder \
    .master("local[1]") \
    .appName("testes_chamadas_abusivas") \
    .config("spark.executor.memory", "512m") \
    .getOrCreate()
spark

## Pastas origem e destino

In [ ]:
# Pasta base origem
INPUT_FOLDER = "/data/cdr/chamadas_abusivas/cdr_processado/Semana86"

# Pasta de arquivos extraídos
EXTRACTED_FOLDER = "/data/cdr/chamadas_abusivas/cdr_extraido/Semana86"

# Pasta de arquivos transformados
TRANSFORMED_FOLDER = "/data/cdr/chamadas_abusivas/cdr_transformado/Semana86"

## Caminho completo dos arquivos

In [ ]:
# Claro/Ericsson
cdr_processado_claro_ericsson = f"{INPUT_FOLDER}/CLARO*Ericsson"
cdr_extraido_claro_ericsson = f"{EXTRACTED_FOLDER}/claro_ericsson_extraido.parquet"
cdr_transformado_claro_ericsson = f"{TRANSFORMED_FOLDER}/claro_ericsson_transformado.parquet"
cdr_ofensores_claro_ericsson = f"{TRANSFORMED_FOLDER}/claro_ericsson_ofensores.parquet"

# Tim/Ericsson
cdr_processado_tim_ericsson = f"{INPUT_FOLDER}/TIM*Ericsson"
cdr_extraido_tim_ericsson = f"{EXTRACTED_FOLDER}/tim_ericsson_extraido.parquet"
cdr_transformado_tim_ericsson = f"{TRANSFORMED_FOLDER}/tim_ericsson_transformado.parquet"
cdr_ofensores_tim_ericsson = f"{TRANSFORMED_FOLDER}/tim_ericsson_ofensores.parquet"

# Vivo/Ericsson
cdr_processado_vivo_ericsson = f"{INPUT_FOLDER}/VIVO*Ericsson"
cdr_extraido_vivo_ericsson = f"{EXTRACTED_FOLDER}/vivo_ericsson_extraido.parquet"
cdr_transformado_vivo_ericsson = f"{TRANSFORMED_FOLDER}/vivo_ericsson_transformado.parquet"
cdr_ofensores_vivo_ericsson = f"{TRANSFORMED_FOLDER}/vivo_ericsson_ofensores.parquet"

# Tim/Volte
cdr_processado_tim_volte = f"{INPUT_FOLDER}/TIM*Volte"
cdr_extraido_tim_volte = f"{EXTRACTED_FOLDER}/tim_volte_extraido.parquet"
cdr_transformado_tim_volte = f"{TRANSFORMED_FOLDER}/tim_volte_transformado.parquet"
cdr_ofensores_tim_volte = f"{TRANSFORMED_FOLDER}/tim_volte_ofensores.parquet"

# Tim/Stir
cdr_processado_tim_stir = f"{INPUT_FOLDER}/TIM*Stir"
cdr_extraido_tim_stir = f"{EXTRACTED_FOLDER}/tim_stir_extraido.parquet"
cdr_transformado_tim_stir = f"{TRANSFORMED_FOLDER}/tim_stir_transformado.parquet"
cdr_ofensores_tim_stir = f"{TRANSFORMED_FOLDER}/tim_stir_ofensores.parquet"

# Tim/Volte
cdr_processado_vivo_volte = f"{INPUT_FOLDER}/VivoVolteConsolidado"
cdr_extraido_vivo_volte = f"{EXTRACTED_FOLDER}/vivo_volte_extraido.parquet"
cdr_transformado_vivo_volte = f"{TRANSFORMED_FOLDER}/vivo_volte_transformado.parquet"
cdr_ofensores_vivo_volte = f"{TRANSFORMED_FOLDER}/vivo_volte_ofensores.parquet"

# Testes

## Extração

In [ ]:
extrator = robocalls.RoboCallsExtractor(spark)
extrator

### Ericsson

In [ ]:
df_extraido_claro_ericsson = extrator.extract_cdr_ericsson(cdr_processado_claro_ericsson, cdr_extraido_claro_ericsson)
df_extraido_claro_ericsson.show(5)
df_extraido_claro_ericsson.printSchema()

In [ ]:
df_extraido_tim_ericsson = extrator.extract_cdr_ericsson(cdr_processado_tim_ericsson, cdr_extraido_tim_ericsson)
df_extraido_tim_ericsson.show(5)

In [ ]:
df_extraido_vivo_ericsson = extrator.extract_cdr_ericsson(cdr_processado_vivo_ericsson, cdr_extraido_vivo_ericsson)
df_extraido_vivo_ericsson.show(5)

### Tim/Stir

In [ ]:
df_extraido_tim_stir = extrator.extract_cdr_tim_stir(cdr_processado_tim_stir,cdr_extraido_tim_stir)
df_extraido_tim_stir.show(5)
df_extraido_tim_stir.printSchema()

### Tim/Volte

In [ ]:
df_extraido_tim_volte = extrator.extract_cdr_tim_volte(cdr_processado_tim_volte,cdr_extraido_tim_volte)
df_extraido_tim_volte.show(5)
df_extraido_tim_volte.printSchema()

### Vivo/Volte

In [ ]:
df_extraido_vivo_volte = extrator.extract_cdr_vivo_volte(cdr_processado_vivo_volte,cdr_extraido_vivo_volte)
df_extraido_vivo_volte.show(5)

In [ ]:
df_extraido_vivo_volte.printSchema()

## Transformação

In [ ]:
transformer = robocalls.RoboCallsTransformer(spark)
transformer

### Ericsson

In [ ]:
df = transformer.transform_cdr_ericsson(cdr_extraido_claro_ericsson, cdr_transformado_claro_ericsson)
df.show(5)
df.printSchema()

In [ ]:
df = transformer.transform_cdr_ericsson(cdr_extraido_tim_ericsson,cdr_transformado_tim_ericsson)
df.show(5)
df.printSchema()

In [ ]:
df = transformer.transform_cdr_ericsson(cdr_extraido_vivo_ericsson,cdr_transformado_vivo_ericsson)
df.show(5)
df.printSchema()

In [ ]:
df = transformer.transform_cdr(cdr_extraido_vivo_ericsson, cdr_transformado_vivo_ericsson, 'ericsson')
df.show(5)
df.printSchema()

### Tim/Volte

In [ ]:
df = transformer.transform_cdr_tim_volte(cdr_extraido_tim_volte,cdr_transformado_tim_volte)
df.show(5)
df.printSchema()

In [ ]:
df.groupBy("chamada_autenticada").agg(F.count("referencia")).show()

### Tim/Stir

In [ ]:
df = transformer.transform_cdr_tim_stir(cdr_extraido_tim_stir,cdr_transformado_tim_stir)
df.show(5)
df.printSchema()

In [ ]:
df = spark.read.parquet(cdr_transformado_tim_stir)
df.show(5)
df.printSchema()

In [ ]:
df.groupBy("chamada_autenticada").agg(F.count("referencia")).show()

### Vivo/Volte

In [ ]:
df = transformer.transform_cdr_vivo_volte(cdr_extraido_vivo_volte,cdr_transformado_vivo_volte)
df.show(5)
df.printSchema()

In [ ]:
df.groupBy("chamada_autenticada").agg(F.count("referencia")).show()

In [ ]:
df.printSchema()

## Cálculo dos ofensores

In [ ]:
agregador = robocalls.RoboCallsAnalyzer(spark)
agregador

### Ericsson

In [ ]:
df_ofensores_claro_ericsson = agregador.analyze(cdr_transformado_claro_ericsson, cdr_ofensores_claro_ericsson)
df_ofensores_claro_ericsson.show(5)

In [ ]:
df_ofensores_tim_ericsson = agregador.analyze(cdr_transformado_tim_ericsson, cdr_ofensores_tim_ericsson)
df_ofensores_tim_ericsson.show(5)

In [ ]:
df_ofensores_vivo_ericsson = agregador.analyze(cdr_transformado_vivo_ericsson, cdr_ofensores_vivo_ericsson)
df_ofensores_vivo_ericsson.show(5)

### Tim/Volte

In [ ]:
df_ofensores_tim_volte = agregador.analyze(cdr_transformado_tim_volte, cdr_ofensores_tim_volte)
df_ofensores_tim_volte.show(5)

### Tim/Stir

In [ ]:
df_ofensores_tim_stir = agregador.analyze(cdr_transformado_tim_stir, cdr_ofensores_tim_stir)
df_ofensores_tim_stir.show(5)

### Vivo/Volte

In [ ]:
df_ofensores_vivo_volte = agregador.analyze(cdr_transformado_vivo_volte, cdr_ofensores_vivo_volte)
df_ofensores_vivo_volte.show(5)